# Diagonally Energy Prediction — 20 Model Comparison

**Dataset**: REFIT Smart Home Dataset — House 1 (Loughborough, UK)  
**Target**: `aggregate_wh` — total daily household energy consumption  
**Test strategy**: 9 strategically chosen days held out — 3 LOW (~7k Wh), 3 MID (~15k Wh), 3 HIGH (~60k Wh) — each from a different month  
**Temperature**: Daily mean and min from Open-Meteo historical API (Loughborough, UK)  
**Metrics**: MAE (Wh), RMSE (Wh), MAPE (%)

In [ ]:
!pip install -q xgboost lightgbm catboost prophet statsforecast

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, StackingRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print('All imports OK')
print('PyTorch:', torch.__version__)

## 1. Load Data
Upload `train.csv` and `test.csv` from `data/processed/` in the repo.

In [ ]:
from google.colab import files
print('Upload train.csv and test.csv')
uploaded = files.upload()

In [ ]:
train = pd.read_csv('train.csv', parse_dates=['datetime'])
test  = pd.read_csv('test.csv',  parse_dates=['datetime'])
print(f'Train: {len(train)} rows  |  Test: {len(test)} rows')
print('Columns:', train.columns.tolist())

In [ ]:
# fetch temperature if not already in the files
if 'temp_mean_c' not in train.columns:
    print('Fetching temperature from Open-Meteo...')
    url = (
        'https://archive-api.open-meteo.com/v1/archive'
        '?latitude=52.77&longitude=-1.20'
        '&start_date=2013-10-09&end_date=2015-07-10'
        '&daily=temperature_2m_mean,temperature_2m_min,temperature_2m_max'
        '&timezone=Europe%2FLondon'
    )
    d = requests.get(url, timeout=30).json()['daily']
    weather = pd.DataFrame({
        'datetime':   pd.to_datetime(d['time']),
        'temp_mean_c': d['temperature_2m_mean'],
        'temp_min_c':  d['temperature_2m_min'],
        'temp_max_c':  d['temperature_2m_max'],
    })
    train = train.merge(weather, on='datetime', how='left')
    test  = test.merge(weather,  on='datetime', how='left')
    print('Temperature merged.')
else:
    print('Temperature already in dataset.')

print(f"Temp range: {train['temp_mean_c'].min():.1f}C to {train['temp_mean_c'].max():.1f}C")

In [ ]:
# combine and clean
all_data = (
    pd.concat([train, test])
      .sort_values('datetime')
      .reset_index(drop=True)
)

# remove sensor-gap zeros, partial last day, and days adjacent to gap (lag=0)
all_data = all_data[
    (all_data['aggregate_wh'] > 5000) &
    (all_data['datetime'] != '2015-07-10') &
    (all_data['lag_1'] > 5000) &
    (all_data['lag_7'] > 5000)
].dropna(subset=['temp_mean_c', 'temp_min_c', 'aggregate_wh']).reset_index(drop=True)

print(f'Clean rows: {len(all_data)}')
print(f'Date range: {all_data["datetime"].min().date()} to {all_data["datetime"].max().date()}')

## 2. Strategic Test Day Selection
9 days held out: 3 LOW, 3 MID, 3 HIGH — each from a different month to avoid seasonal clustering.

In [ ]:
TEST_DATES = pd.to_datetime([
    # LOW consumption ~7,000 Wh
    '2013-10-16',   # Oct
    '2014-12-07',   # Dec
    '2015-01-03',   # Jan
    # MID consumption ~15,000 Wh
    '2014-07-21',   # Jul
    '2014-11-12',   # Nov
    '2014-08-01',   # Aug
    # HIGH consumption ~60,000 Wh
    '2013-11-22',   # Nov
    '2013-12-12',   # Dec
    '2014-01-19',   # Jan
])

test_df  = all_data[all_data['datetime'].isin(TEST_DATES)].copy().reset_index(drop=True)
train_df = all_data[~all_data['datetime'].isin(TEST_DATES)].copy().reset_index(drop=True)

def band(wh):
    return 'LOW' if wh < 12000 else ('MID' if wh < 25000 else 'HIGH')

test_df['band'] = test_df['aggregate_wh'].apply(band)

display_df = test_df[['band','datetime','temp_mean_c','aggregate_wh']].copy()
display_df.columns = ['Band','Date','Temp C','Actual Wh']
display_df['Date'] = display_df['Date'].dt.strftime('%Y-%m-%d')
display_df['Actual Wh'] = display_df['Actual Wh'].round(0).astype(int)
print(display_df.to_string(index=False))
print(f'\nTraining rows: {len(train_df)}  |  Test rows: {len(test_df)}')

## 3. Feature Setup & Helpers

In [ ]:
FEATURES = [
    'day_of_week', 'month', 'is_weekend',
    'lag_1', 'lag_7', 'rolling_mean_7',
    'heater_lag_1', 'heater_lag_7', 'heater_rolling_mean_7',
    'temp_mean_c', 'temp_min_c',
]

X_train = train_df[FEATURES].values
y_train = train_df['aggregate_wh'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['aggregate_wh'].values

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_train)
X_te_s   = scaler.transform(X_test)

leaderboard = []

def show_results(name, y_pred):
    y_pred = np.array(y_pred)
    df = test_df[['band','datetime','temp_mean_c','aggregate_wh']].copy()
    df['predicted_wh'] = y_pred.round(0).astype(int)
    df['error_pct']    = ((df['predicted_wh'] - df['aggregate_wh']).abs() / df['aggregate_wh'] * 100).round(1)
    df.columns         = ['Band','Date','Temp C','Actual Wh','Predicted Wh','Error %']
    df['Date']         = df['Date'].dt.strftime('%Y-%m-%d')
    df['Actual Wh']    = df['Actual Wh'].round(0).astype(int)
    df['Temp C']       = df['Temp C'].round(1)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    print(f"\n{'='*68}")
    print(f"  {name}")
    print(f"{'='*68}")
    print(df.to_string(index=False))
    print(f"\n  MAE: {mae:,.0f} Wh   RMSE: {rmse:,.0f} Wh   MAPE: {mape:.1f}%")
    leaderboard.append({'Model': name, 'MAE_Wh': round(mae), 'RMSE_Wh': round(rmse), 'MAPE_%': round(mape,1)})

print('Features:', FEATURES)
print('X_train shape:', X_train.shape, '  X_test shape:', X_test.shape)

## 4. Tree-Based Models (1–6)
These models use the full feature matrix including temperature and lag features.

In [ ]:
# ── 1. XGBoost ────────────────────────────────────────────────────────────────
m = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                 subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
m.fit(X_train, y_train)
show_results('1. XGBoost', m.predict(X_test))

In [ ]:
# ── 2. LightGBM ───────────────────────────────────────────────────────────────
m = LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
m.fit(X_train, y_train)
show_results('2. LightGBM', m.predict(X_test))

In [ ]:
# ── 3. CatBoost ───────────────────────────────────────────────────────────────
m = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=5,
                      random_seed=42, verbose=0)
m.fit(X_train, y_train)
show_results('3. CatBoost', m.predict(X_test))

In [ ]:
# ── 4. Random Forest ──────────────────────────────────────────────────────────
m = RandomForestRegressor(n_estimators=300, max_depth=10,
                           random_state=42, n_jobs=-1)
m.fit(X_train, y_train)
show_results('4. Random Forest', m.predict(X_test))

In [ ]:
# ── 5. Extra Trees ────────────────────────────────────────────────────────────
m = ExtraTreesRegressor(n_estimators=300, max_depth=10,
                         random_state=42, n_jobs=-1)
m.fit(X_train, y_train)
show_results('5. Extra Trees', m.predict(X_test))

In [ ]:
# ── 6. Gradient Boosting (sklearn) ────────────────────────────────────────────
m = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=4, subsample=0.8, random_state=42)
m.fit(X_train, y_train)
show_results('6. Gradient Boosting', m.predict(X_test))

## 5. Linear Models (7–10)
Features are standardised (zero mean, unit variance) before fitting.

In [ ]:
# ── 7. Ridge Regression ───────────────────────────────────────────────────────
m = Ridge(alpha=10.0)
m.fit(X_tr_s, y_train)
show_results('7. Ridge Regression', m.predict(X_te_s))

In [ ]:
# ── 8. Lasso Regression ───────────────────────────────────────────────────────
m = Lasso(alpha=500.0, max_iter=10000)
m.fit(X_tr_s, y_train)
show_results('8. Lasso Regression', m.predict(X_te_s))

In [ ]:
# ── 9. ElasticNet ─────────────────────────────────────────────────────────────
m = ElasticNet(alpha=500.0, l1_ratio=0.5, max_iter=10000)
m.fit(X_tr_s, y_train)
show_results('9. ElasticNet', m.predict(X_te_s))

In [ ]:
# ── 10. Huber Regressor (robust to outliers) ──────────────────────────────────
# Good for this dataset: heater spikes on cold days look like outliers to linear models
m = HuberRegressor(epsilon=1.5, max_iter=500)
m.fit(X_tr_s, y_train)
show_results('10. Huber Regressor', m.predict(X_te_s))

## 6. Other ML Models (11–13)

In [ ]:
# ── 11. SVR (RBF kernel) ──────────────────────────────────────────────────────
m = SVR(kernel='rbf', C=1e5, gamma='scale', epsilon=500)
m.fit(X_tr_s, y_train)
show_results('11. SVR (RBF)', m.predict(X_te_s))

In [ ]:
# ── 12. K-Nearest Neighbours ──────────────────────────────────────────────────
m = KNeighborsRegressor(n_neighbors=7, weights='distance', metric='euclidean')
m.fit(X_tr_s, y_train)
show_results('12. KNN Regressor', m.predict(X_te_s))

In [ ]:
# ── 13. MLP (sklearn) ─────────────────────────────────────────────────────────
m = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=500,
    random_state=42,
    learning_rate_init=0.001,
    early_stopping=True,
    validation_fraction=0.1,
)
m.fit(X_tr_s, y_train)
show_results('13. MLP (sklearn)', m.predict(X_te_s))

## 7. Statistical Time Series Models (14–17)
These models use **rolling-origin evaluation**: for each test day, the model is fitted on all data
before that date (excluding earlier test dates) and predicts one step ahead.

In [ ]:
# ── 14. Prophet ───────────────────────────────────────────────────────────────
try:
    from prophet import Prophet

    preds = []
    for dt in TEST_DATES:
        history = all_data[
            (all_data['datetime'] < dt) & (~all_data['datetime'].isin(TEST_DATES))
        ][['datetime', 'aggregate_wh']].copy()
        history.columns = ['ds', 'y']

        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
        )
        m.fit(history)
        future = pd.DataFrame({'ds': [dt]})
        preds.append(max(m.predict(future)['yhat'].values[0], 0))

    show_results('14. Prophet', np.array(preds))

except Exception as e:
    print(f'Prophet error: {e}')
    leaderboard.append({'Model': '14. Prophet', 'MAE_Wh': None, 'RMSE_Wh': None, 'MAPE_%': None})

In [ ]:
# ── 15. SARIMAX (with temperature as exogenous) ───────────────────────────────
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    preds = []
    for dt in TEST_DATES:
        history = all_data[
            (all_data['datetime'] < dt) & (~all_data['datetime'].isin(TEST_DATES))
        ].copy()
        endog       = history['aggregate_wh'].values
        exog_train  = history[['temp_mean_c', 'temp_min_c']].values
        exog_pred   = all_data[all_data['datetime'] == dt][['temp_mean_c', 'temp_min_c']].values

        m   = SARIMAX(endog, exog=exog_train, order=(1, 0, 1),
                      seasonal_order=(1, 0, 1, 7),
                      enforce_stationarity=False, enforce_invertibility=False)
        fit = m.fit(disp=False)
        preds.append(max(fit.forecast(steps=1, exog=exog_pred)[0], 0))

    show_results('15. SARIMAX', np.array(preds))

except Exception as e:
    print(f'SARIMAX error: {e}')
    leaderboard.append({'Model': '15. SARIMAX', 'MAE_Wh': None, 'RMSE_Wh': None, 'MAPE_%': None})

In [ ]:
# ── 16. Holt-Winters Triple Exponential Smoothing ─────────────────────────────
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

    preds = []
    for dt in TEST_DATES:
        history = all_data[
            (all_data['datetime'] < dt) & (~all_data['datetime'].isin(TEST_DATES))
        ]['aggregate_wh'].values

        m   = ExponentialSmoothing(history, trend='add', seasonal='add', seasonal_periods=7)
        fit = m.fit(optimized=True)
        preds.append(max(fit.forecast(1)[0], 0))

    show_results('16. Holt-Winters', np.array(preds))

except Exception as e:
    print(f'Holt-Winters error: {e}')
    leaderboard.append({'Model': '16. Holt-Winters', 'MAE_Wh': None, 'RMSE_Wh': None, 'MAPE_%': None})

In [ ]:
# ── 17. Theta (statsforecast) ─────────────────────────────────────────────────
try:
    from statsforecast import StatsForecast
    from statsforecast.models import Theta

    preds = []
    for dt in TEST_DATES:
        history = all_data[
            (all_data['datetime'] < dt) & (~all_data['datetime'].isin(TEST_DATES))
        ][['datetime', 'aggregate_wh']].copy()
        history['unique_id'] = 'house1'
        history.columns = ['ds', 'y', 'unique_id']

        sf  = StatsForecast(models=[Theta(season_length=7)], freq='D', n_jobs=1)
        sf.fit(history[['unique_id', 'ds', 'y']])
        pred_df = sf.predict(h=1)
        preds.append(max(float(pred_df['Theta'].values[0]), 0))

    show_results('17. Theta', np.array(preds))

except Exception as e:
    print(f'Theta error: {e}')
    leaderboard.append({'Model': '17. Theta', 'MAE_Wh': None, 'RMSE_Wh': None, 'MAPE_%': None})

## 8. Neural Sequence Models (18–19)
Both models use a sliding window of the last **30 days** of `aggregate_wh` as input sequence.
Trained on sequences from non-test days; predict each test day using its preceding 30-day window.

In [ ]:
# shared sequence preparation
SEQ_LEN = 30

# normalise using training series stats
train_series = train_df['aggregate_wh'].values.astype(np.float32)
S_MEAN, S_STD = train_series.mean(), train_series.std()

# build input sequences from training data only
train_norm = (train_series - S_MEAN) / S_STD

def make_sequences(series, seq_len):
    X, y = [], []
    for i in range(seq_len, len(series)):
        X.append(series[i-seq_len:i])
        y.append(series[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_seq, y_seq = make_sequences(train_norm, SEQ_LEN)
loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_seq).unsqueeze(-1), torch.FloatTensor(y_seq)),
    batch_size=32, shuffle=True
)

def get_input_window(dt):
    preceding = all_data[
        (all_data['datetime'] < dt) & (~all_data['datetime'].isin(TEST_DATES))
    ]['aggregate_wh'].values[-SEQ_LEN:]
    if len(preceding) < SEQ_LEN:
        return None
    norm = (preceding.astype(np.float32) - S_MEAN) / S_STD
    return torch.FloatTensor(norm).unsqueeze(0).unsqueeze(-1)

criterion = nn.MSELoss()
print(f'Sequence shape: {X_seq.shape}  |  S_MEAN={S_MEAN:.0f}  S_STD={S_STD:.0f}')

In [ ]:
# ── 18. LSTM (PyTorch) ────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

lstm      = LSTMModel()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.001)

for epoch in range(60):
    lstm.train()
    for xb, yb in loader:
        pred = lstm(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1}/60  loss={loss.item():.4f}')

lstm.eval()
preds = []
with torch.no_grad():
    for dt in TEST_DATES:
        x_in = get_input_window(dt)
        if x_in is None:
            preds.append(S_MEAN)
            continue
        p = lstm(x_in).item() * S_STD + S_MEAN
        preds.append(max(p, 0))

show_results('18. LSTM', np.array(preds))

In [ ]:
# ── 19. 1D CNN (PyTorch) ──────────────────────────────────────────────────────
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.net(x).squeeze(-1)
        return self.fc(x).squeeze()

cnn       = CNNModel()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

for epoch in range(60):
    cnn.train()
    for xb, yb in loader:
        pred = cnn(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1}/60  loss={loss.item():.4f}')

cnn.eval()
preds = []
with torch.no_grad():
    for dt in TEST_DATES:
        x_in = get_input_window(dt)
        if x_in is None:
            preds.append(S_MEAN)
            continue
        p = cnn(x_in).item() * S_STD + S_MEAN
        preds.append(max(p, 0))

show_results('19. 1D CNN', np.array(preds))

## 9. Stacking Ensemble (20)
Four tree models as base learners, Ridge as the meta-learner.  
Predictions from each base model become the features the meta-learner trains on.

In [ ]:
# ── 20. Stacking Ensemble ─────────────────────────────────────────────────────
base_estimators = [
    ('xgb',  XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0)),
    ('lgb',  LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)),
    ('rf',   RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
    ('cat',  CatBoostRegressor(iterations=150, random_seed=42, verbose=0)),
]

stack = StackingRegressor(
    estimators=base_estimators,
    final_estimator=Ridge(alpha=10.0),
    cv=5,
    n_jobs=-1,
)
stack.fit(X_train, y_train)
show_results('20. Stacking Ensemble', stack.predict(X_test))

## 10. Final Leaderboard

In [ ]:
lb = pd.DataFrame(leaderboard).dropna()
lb = lb.sort_values('MAPE_%').reset_index(drop=True)
lb.index = lb.index + 1
lb['MAE_Wh']  = lb['MAE_Wh'].astype(int)
lb['RMSE_Wh'] = lb['RMSE_Wh'].astype(int)

print('\n' + '='*60)
print('  FINAL LEADERBOARD  (sorted by MAPE — lower is better)')
print('='*60)
print(lb.to_string())
print()
print(f'  WINNER : {lb.iloc[0]["Model"]}')
print(f'  MAPE   : {lb.iloc[0]["MAPE_%"]}%')
print(f'  MAE    : {lb.iloc[0]["MAE_Wh"]:,} Wh')
print(f'  RMSE   : {lb.iloc[0]["RMSE_Wh"]:,} Wh')

## 11. Why These Models for This Problem

| Family | Best suited because |
|---|---|
| **XGBoost / LightGBM / CatBoost** | Handle non-linear temperature–heater interaction, robust to the right-skewed heater distribution, work well on small tabular datasets (~500 rows) |
| **Random Forest / Extra Trees** | Ensemble averaging reduces variance from heater spike days; no hyperparameter tuning needed |
| **Gradient Boosting** | Sequential error correction is effective when residuals are dominated by a few extreme winter days |
| **Ridge / Huber** | Huber is robust to the extreme outliers (winter HIGH days); Ridge gives a calibrated linear baseline |
| **SVR** | RBF kernel can capture the non-linear temperature relationship; good on small datasets |
| **MLP** | Learns feature interactions without manual engineering; early stopping prevents overfitting |
| **Prophet** | Automatically learns yearly and weekly seasonality; the additive/multiplicative mode switch suits this dataset |
| **SARIMAX** | Seasonal ARIMA with temperature as exogenous variable directly models the heater–weather link |
| **Holt-Winters** | Triple exponential smoothing with 7-day seasonal period captures weekly routine well |
| **Theta** | Performs well on short series; decomposes trend and seasonality without overfitting |
| **LSTM** | Memory cells retain patterns across 30-day windows; learns transition between heating seasons |
| **1D CNN** | Captures local temporal patterns (e.g. 3-day warm spells) efficiently via convolution |
| **Stacking** | Combines the strengths of tree models; the meta-learner corrects systematic biases of individual models |

**Key metric to use: MAPE** — it normalises error by actual consumption, making HIGH days (60k Wh) and LOW days (7k Wh) contribute equally to the score rather than HIGH days dominating MAE/RMSE.